# Neural Language Models: CNNs, RNNs, and Training Mechanisms

Neural Language Models (NLMs) address the fundamental limitations of Statistical Language Models (n-gram models). This module covers the transition from statistical methods to neural architectures, specifically exploring Convolutional Neural Networks (CNNs) and Recurrent Neural Networks (RNNs) for language modeling, along with their training dynamics.

## 1. Limitations of Statistical Language Models

Statistical language models predict the next word by sampling from a probability distribution based on counts (e.g., unigram, bigram, or trigram).

* **Sparsity:** The n-gram matrices (like the bigram matrix) are highly sparse (filled with zeros), making it difficult to store and compute efficiently.
* **Out-of-Vocabulary (OOV) Words:** They struggle significantly with unknown words that did not appear in the training corpus.

Neural Language Models (first proposed around 2001–2002 by Yoshua Bengio) solve these issues by operating on dense word embeddings (e.g., Word2Vec, GloVe) rather than discrete counts.

---

## 2. CNN-Based Neural Language Models

To use a CNN for language modeling, we treat a sentence as a 1D sequence of tokens. Instead of applying a 2D filter over image pixels, we apply a 1D filter over a sequence of word embeddings.

### Architecture and Mechanism

1. **Window Size (Context):** Decide on a fixed window size (e.g., 4 previous words).
2. **Input Flattening:** Retrieve the embedding for each word. If the window size is 4 and each embedding is of size 4, concatenate them into a flattened $1 \times 16$ vector. Let this be $X = [x_1, x_2, x_3, x_4]$.
3. **Applying the Kernel (W):** Multiply the flattened input by a weight matrix (kernel) $W$. If $W$ is of size $16 \times 8$, the output hidden state will be a $1 \times 8$ vector.
4. **Projection (U):** Pass the hidden state through a feedforward linear layer $U$ (e.g., of size $8 \times \vert{}V\vert{}$, where $\vert{}V\vert{}$ is the vocabulary size, say 1 million).
5. **Prediction:** Apply the `softmax` function to the output vector to convert the raw logits into a valid probability distribution. Sample the word with the highest probability.

### Advantages of CNN LMs

* **No Sparsity:** Dense embeddings eliminate the matrix sparsity problem.
* **Handles Unknown Words:** Systematically processes unknown words via their nearest embedding space representations.
* **Memory Efficient (Partially):** Eliminates the need to store massive bigram/trigram transition matrices.

### Limitations of CNN LMs

1. **Fixed Context Window:** Cannot capture infinite or very long context. You must define a strict cutoff (hyperparameter).
2. **Parameter Scaling:** As the window size increases, the input vector size increases, meaning the weight matrix $W$ must also increase in dimensions.
3. **No Message Passing Across Positions (Disjoint Learning):**
* When multiplying the flattened embedding matrix $[E_1, E_2, E_3, E_4]$ with $W$, the math causes $E_1$ to strictly interact with the first chunk of $W$, $E_2$ with the second, etc.
* Information from $E_1$ does not pass through the columns associated with $E_2$. Essentially, the network learns independent functions for each positional slot rather than synthesizing sequential dependencies globally.



---

## 3. RNN-Based Neural Language Models

Recurrent Neural Networks (RNNs) solve the fixed-window limitation of CNNs by processing inputs sequentially and maintaining a persistent hidden state.

### Architecture and Mechanism

At any time step $t$, the RNN computes a hidden state $h_t$ using the current input $x_t$ (word embedding) and the *previous* hidden state $h_{t-1}$.

$$h_t = f(h_{t-1}W_H + x_tW_E + b)$$

Where:

* $x_t$: Current word embedding.
* $h_{t-1}$: Hidden state from the previous time step (starts with a dummy $h_0$).
* $W_H$: Weight matrix for the hidden state projection.
* $W_E$: Weight matrix for the input embedding projection.
* $b$: Bias term.
* $f$: A non-linear activation function (e.g., $\sigma$ or $\tanh$).

The output prediction follows the same logic as the CNN: pass $h_t$ through a projection matrix $U$ and apply softmax to get vocabulary probabilities.

### CNN vs. RNN for Language Modeling

| Feature | CNN Language Model | RNN Language Model |
| --- | --- | --- |
| **Context Size** | Fixed (hyperparameter limit). | Theoretically infinite (captures all previous steps). |
| **Parameter Scaling** | $W$ grows as window size increases. | $W_H$ and $W_E$ are **shared** and remain fixed regardless of sequence length. |
| **Position Symmetry** | Disjoint processing based on position. | Symmetric; the same $W_E$ applies to all embeddings fairly. |
| **Speed** | Fast (can be parallelized easily). | Slow (sequential computation; $h_t$ requires $h_{t-1}$). |

---

## 4. Training the RNN

### Objective Function and Cross-Entropy Loss

The goal is to predict the correct next word at *every* time step.
If the input sequence is "the students opened their", the targets are:

* Input "the" $\rightarrow$ Target "students"
* Input "students" $\rightarrow$ Target "opened"

We use **Cross-Entropy Loss** to measure the error between the predicted probability distribution ($\hat{y}$) and the actual ground truth one-hot encoded vector ($y$).

$$J_t(\theta) = -\log(\hat{y}_c)$$


*(where $\hat{y}_c$ is the predicted probability for the actual correct word at step $t$)*

The total loss is the average across all time steps:


$$\text{Total Loss} = \frac{1}{T} \sum_{t=1}^{T} J_t(\theta)$$

### Teacher Forcing

During training, the model might predict an incorrect word (e.g., "teachers" instead of "students").

* **Standard Autoregression:** Feeding the *predicted* word ("teachers") into the next time step. This causes compounding errors during early training.
* **Teacher Forcing:** Feeding the *ground truth* previous word ("students") into the next time step, regardless of what the model actually predicted. This is standard practice to stabilize and speed up RNN training.

---

## 5. Backpropagation Through Time (BPTT)

To update the parameters ($W_H$, $W_E$, $U$), we must calculate the gradients of the Loss with respect to those weights. The hardest parameter to update is $W_H$ (referred to as $W$ in the derivation) because it is used repeatedly at every time step.

Using the multivariable chain rule, the gradient of the loss at time $t$ ($L_t$) with respect to $W$ requires summing the gradients across all previous time steps $k$:

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial h_t} \sum_{k=0}^{t} \left( \frac{\partial h_t}{\partial h_k} \frac{\partial h_k}{\partial W} \right)$$

### The Gradient Problems

The term $\frac{\partial h_t}{\partial h_k}$ is a chain of derivatives (e.g., $\frac{\partial h_4}{\partial h_1} = \frac{\partial h_4}{\partial h_3} \times \frac{\partial h_3}{\partial h_2} \times \frac{\partial h_2}{\partial h_1}$). Since these are matrix multiplications, we must evaluate the eigenvalues of the weight matrix.

1. **Vanishing Gradient Problem:**
If the largest eigenvalue of the weight matrix is less than 1, multiplying many of these small matrices together drives the gradient to zero. The network fails to update weights for long-term dependencies, effectively "forgetting" early words in a sequence.
2. **Exploding Gradient Problem:**
If the largest eigenvalue is greater than 1, the gradients grow exponentially, causing numerical instability (NaNs) and model failure.

### Solutions

* **Gradient Clipping:** Caps the maximum value of gradients at a specific threshold. This directly solves the **exploding gradient** problem.
* **Truncated BPTT:** Instead of backpropagating all the way to $t=0$, we stop backpropagating after a fixed number of steps.
* **Advanced Architectures:** Standard RNNs are highly susceptible to vanishing gradients. This is solved by using gated architectures like **LSTM** (Long Short-Term Memory) or **GRU** (Gated Recurrent Unit).

---

## Practice Questions

**Q1. In a CNN-based language model, what happens to the weight matrix $W$ if you increase the context window size?**
A) Its dimensions decrease to accommodate the larger context.
B) Its dimensions increase, meaning more parameters to learn.
C) The dimensions remain exactly the same.
D) The matrix becomes sparse.
*Answer: B. The input vector concatenates more word embeddings, thus requiring a larger weight matrix to project it.*

**Q2. Which of the following is a key advantage of RNNs over CNNs for language modeling?**
A) RNNs can compute sequences in parallel, making them much faster.
B) The size of the weight matrix in an RNN is independent of the context length.
C) RNNs are immune to the vanishing gradient problem.
D) RNNs do not require embeddings to process words.
*Answer: B. In an RNN, $W_H$ and $W_E$ are shared across all time steps. Their dimensions depend only on the embedding size and hidden state size, not the length of the sequence.*

**Q3. During the training of an RNN language model, the model predicts the word "closed" instead of the actual target "opened". For the next time step, the system feeds the word "opened" as the input anyway. What is this technique called?**
A) Gradient Clipping
B) Backpropagation Through Time
C) Truncated BPTT
D) Teacher Forcing
*Answer: D. Teacher forcing involves passing the actual ground truth output as the next input during training.*

**Q4. A recurrent neural network is suffering from the exploding gradient problem during BPTT. Which of the following is the most direct solution?**
A) Gradient clipping
B) Using a unigram model
C) Increasing the window size
D) Teacher forcing
*Answer: A. Gradient clipping sets a strict threshold to prevent gradients from reaching unstable, massive values.*

---

## Revision Summary

* **Stats LMs vs Neural LMs:** Neural models use dense embeddings, solving sparsity and out-of-vocabulary issues inherent to n-gram matrices.
* **CNN limitations:** Fixed window limits context scope; parameter size scales with window size; positional weights are disjoint so no sequential message passing occurs across embedding columns.
* **RNN Architecture:** Utilizes shared weight matrices ($W_E, W_H$) across all time steps. Capable of theoretically processing infinite context.
* **Training:** Uses Cross-Entropy loss at each sequence step. Relies on **Teacher Forcing** (passing ground truth as next input) for stable training.
* **BPTT:** Updating recurring weights requires the multivariable chain rule spanning all previous steps.
* **Gradient decay:** Matrix multiplications during BPTT lead to **vanishing gradients** (eigenvalues $<1$, earlier steps forgotten) or **exploding gradients** (eigenvalues $>1$, fixed via clipping).